# Análise Exploratória — Ensino Médio Público da Paraíba

**Disciplina: Análise de Dados**  
**Recortes:** ENEM e Censo Escolar 2024; IDEB 2025.  
**Metodologia:** D.E.I.L.A.S.

Este notebook reproduz os indicadores de **relatórios agregados verificados** e gera gráficos, sem exigir o download dos microdados com registros individuais. Não substitui a análise integral das bases.

## 1. Fontes e limitações

- [Perfil ENEM/Censo 2024](../reports/inspecao/perfil_paraiba.md)
- [Auditoria das chaves ENEM–Censo 2024](../reports/filtragem/validacao_chaves_2024.md)
- [IDEB 2025 por escola](../reports/inspecao/ideb_pb_2025.md)
- [Relatório exploratório](../reports/analise_exploratoria/analise_preliminar.md)

**Atenção:** participantes do ENEM não representam todos os estudantes da rede. O local de prova não equivale à escola. IDEB 2025 e Censo 2024 são anos distintos. Não interpretar relações descritivas como causalidade.

In [ ]:
from pathlib import Path
import importlib.util

raiz = Path.cwd().resolve()
while raiz != raiz.parent and not (raiz / 'src' / '05_exploracao_preliminar.py').is_file():
    raiz = raiz.parent
script = raiz / 'src' / '05_exploracao_preliminar.py'
if not script.is_file():
    raise FileNotFoundError('Abra o notebook na pasta do repositório projeto_inep.')
spec = importlib.util.spec_from_file_location('eda_prel', script)
eda = importlib.util.module_from_spec(spec)
spec.loader.exec_module(eda)

dados = eda.carregar()
assert eda.validar(dados)
print('Indicadores carregados e validados com sucesso.')

## 2. Oferta e infraestrutura escolar (Censo 2024)

A unidade aqui é a **escola estadual com oferta de ensino médio na PB**. Valores `1`, `0` e vazios foram preservados como categorias distintas.

In [ ]:
censo = dados['censo']
escolas = censo['escolas_estaduais_ensino_medio_PB']
print(f'Escolas estaduais com ensino médio: {escolas}')
print(f'Municípios atendidos: {censo["municipios_estaduais_EM"]}')
print(f'Matrículas declaradas: {censo["matriculas_ensino_medio_somadas"]:,}')

for campo, rotulo in eda.INFRA_LABELS.items():
    v = censo['valores_infraestrutura'][campo]
    print(f'{rotulo}: {v.get("1", 0)} de {escolas} ({eda.percentual(v.get("1", 0), escolas):.1f}%) — não informado: {v.get("(vazio)", 0)}')

## 3. Cobertura do relacionamento por escola

O cruzamento utiliza os códigos `CO_ESCOLA` (ENEM) e `CO_ENTIDADE` (Censo) referentes a **2024**. É um vínculo entre escolas; não associa notas ao questionário individual.

In [ ]:
a = dados['auditoria']
print('Escolas ENEM identificadas:', a['escolas_enem'])
print('Escolas Censo:', a['escolas_censo'])
print('Escolas pareadas:', a['escolas_pareadas'])
print('Cobertura das escolas Censo:', eda.percentual(a['escolas_pareadas'], a['escolas_censo']), '%')
print('Cobertura dos registros ENEM:', eda.percentual(a['linhas_pareadas'], a['linhas_enem']), '%')

## 4. Notas médias ENEM 2024

As médias usam somente registros com escola estadual da PB identificada e nota válida. Cada área possui um denominador próprio e não deve ser tratada automaticamente como medida comparável de qualidade escolar.

In [ ]:
notas = dados['resultados']['notas_de_participantes_com_escola_estadual_PB']
for campo, rotulo in eda.ENEM_LABELS.items():
    info = notas[campo]
    print(f'{rotulo}: média={info["media"]:.2f} | notas válidas={info["validas"]:,}')

## 5. IDEB 2025 por escola

Média **aritmética simples** calculada apenas entre escolas estaduais com nota válida. Não substitui o IDEB oficial da rede.

In [ ]:
ideb = dados['ideb']
print('Escolas estaduais da PB na planilha:', ideb['total'])
print('Com nota válida:', ideb['validas'])
print('Sem nota válida:', ideb['sem_nota'])
print('Média simples:', ideb['media_simples'])
print('Mediana:', ideb['mediana'])

## 6. Gerar visualizações

A execução abaixo reproduz as figuras SVG com os dados agregados, e o relatório Markdown correspondente.

In [ ]:
arquivo = eda.gerar(dados)
print('Relatório atualizado:', arquivo)
print('Figura infraestrutura:', raiz / 'reports/figuras/infraestrutura_censo_PB_2024.svg')
print('Figura cobertura:', raiz / 'reports/figuras/cobertura_chaves_PB_2024.svg')

from IPython.display import SVG, display
display(SVG(filename=str(raiz / 'reports/figuras/infraestrutura_censo_PB_2024.svg')))
display(SVG(filename=str(raiz / 'reports/figuras/cobertura_chaves_PB_2024.svg')))

## 7. Próximos passos

1. Analisar a infraestrutura por município usando bases escolares limpas e chaves auditadas.
2. Estudar dispersões, medianas e comparações de notas válidas entre grupos adequados.
3. Incorporar ENEM 2025 e Censo 2025 **após** validar seus cabeçalhos e dicionários.
4. Preparar indicadores executivos e recomendações sujeitas à validação de contexto, orçamento e impacto.

**Não fazer:** juntar arquivos ENEM de participantes e resultados por posição; confundir local de prova com escola; apresentar média simples do IDEB como resultado oficial da rede.